## Includes

In [1]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import torch.utils.data as data
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger

from vv.utilities.utilities import get_dir

## Model setup

In [2]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Sequential(nn.Linear(28 * 28, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, x):
        return self.l1(x)


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 28 * 28))

    def forward(self, x):
        return self.l1(x)

In [3]:
class LitAutoEncoder(L.LightningModule):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

        # This attribute makes the summary display the layer sizes
        self.example_input_array = torch.Tensor(32, 1, 28, 28)

        # Saves hyperparameters that are passed to the __init__ method
        self.save_hyperparameters(ignore=['encoder', 'decoder'])

    def forward(self, x):
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        return self.decoder(z)

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, y = batch
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        x_hat = self.decoder(z)
        loss = F.mse_loss(x_hat, x)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        # this is the validation loop
        x, y = batch
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        x_hat = self.decoder(z)
        val_loss = F.mse_loss(x_hat, x)
        self.log("val_loss", val_loss, prog_bar=True)
    
    def test_step(self, batch, batch_idx):
        # this is the test loop
        x, y = batch
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        x_hat = self.decoder(z)
        test_loss = F.mse_loss(x_hat, x)
        self.log("test_loss", test_loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

## Data setup

In [4]:
# Load data sets
transform = transforms.ToTensor()
train_set = MNIST(root=get_dir("data"), download=True, train=True, transform=transform);
test_set = MNIST(root=get_dir("data"), download=True, train=False, transform=transform);

# use 20% of training data for validation
train_set_size = int(len(train_set) * 0.8)
valid_set_size = len(train_set) - train_set_size

# split the train set into two
seed = torch.Generator().manual_seed(42)
train_set, valid_set = data.random_split(train_set, [train_set_size, valid_set_size], generator=seed)

In [5]:
# model
model = LitAutoEncoder(Encoder(), Decoder())

train_loader = DataLoader(train_set, batch_size=5000, 
                          num_workers=10)
valid_loader = DataLoader(valid_set, batch_size=5000,
                          num_workers=10)
test_loader = DataLoader(test_set, batch_size=5000,
                         num_workers=10)

## Training process

Tensorboard in VSC

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir=lightning_logs/

Tensorboard in Browser

In [ ]:
tensorboard --logdir=models/test_model/version0

In [6]:
logger = TensorBoardLogger(save_dir=get_dir("models", create=True),
                           name="test_model",
                           log_graph=True)

In [7]:
# train model with validation
trainer = L.Trainer(default_root_dir=get_dir("models", create=True),
                    max_epochs=300,
                    enable_checkpointing=True,
                    logger=logger)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


### Using the profiler

In [ ]:
from lightning.pytorch.callbacks import DeviceStatsMonitor

# train model with validation
trainer = L.Trainer(default_root_dir=get_dir("models", create=True),
                    max_epochs=500,
                    enable_checkpointing=True,
                    profiler="simple",
                    logger=logger,
                    callbacks=[DeviceStatsMonitor()])

### Train

In [8]:
torch.set_float32_matmul_precision('high')

In [9]:
trainer.fit(model=model,
            train_dataloaders=train_loader,
            val_dataloaders=valid_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type    | Params | In sizes  | Out sizes
------------------------------------------------------------
0 | encoder | Encoder | 50.4 K | [32, 784] | [32, 3]  
1 | decoder | Decoder | 51.2 K | [32, 3]   | [32, 784]
------------------------------------------------------------
101 K     Trainable params
0         Non-trainable params
101 K     Total params
0.407     Total estimated model params size (MB)


/home/computerman/Desktop/VQ-VAE/VQ-VAE-Praktikum-SS24-HD/venv/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 299: 100%|██████████| 10/10 [00:01<00:00,  6.61it/s, v_num=5, val_loss=0.0382]

`Trainer.fit` stopped: `max_epochs=300` reached.


Epoch 299: 100%|██████████| 10/10 [00:01<00:00,  6.58it/s, v_num=5, val_loss=0.0382]


### Load model

In [ ]:
model_loaded = LitAutoEncoder.load_from_checkpoint(
    checkpoint_path=get_dir("models",
                            "lightning_logs",
                            "version_9",
                            "checkpoints",
                            "epoch=299-step=3000.ckpt"),
    # The following arguments are only necessary if hyperparams of model are not saved
    # larger stuff like nn modules should not be saved
    # In this case they need to be passed as arguments to the load_from_checkpoint method
    encoder=Encoder(),
    decoder=Decoder()
)

## Testing

In [ ]:
model_loaded.eval()
trainer.test(model_loaded, dataloaders=test_loader)